# Agentic Search with Shell tool

This notebook walks you through an agentic search example with a shell tool that lets the agent run commands in the terminal to navigate a local filesystem and use `grep` to search files. It builds on the first implementation by adding `jina-grep-cli` as a semantic search alternative to `grep`.

This approach lets the agent pull context stored in a local filesystem into its context window to answer **user queries containing keywords** and handle **conceptual questions**.

<img src="../img/Agentic search with shell tool.png" width="1000" alt="Agentic search with shell tool" />

## Preparation

Make sure you have completed the setup instructions according to the README.md.

Load environment variables from `.env` file, such as API keys.

In [21]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

LITELLM_API_KEY = os.getenv("LITELLM_API_KEY")
LITELLM_API_BASE = os.getenv("LITELLM_API_BASE")

## Set up agent with shell tool (and `grep`)

### Configure the LLM

Set up your LLM with the right parameters for your use case:
Below, we're using OpenAI's models through LiteLLM but you can switch it out with any model capable of tool use.

In [22]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_base=LITELLM_API_BASE,
    api_key=LITELLM_API_KEY,
    model="llm-gateway/gpt-5.4-nano",
    temperature=0.5,
)

### Define the system prompt

The system prompt defines your agent’s role and behavior. 

We also include information about what out-of-context knowledge source it has. In this case it has access to a **local filesystem** with the conference data.

In [23]:
from IPython.display import display, Markdown

with open("system_prompt_fs.md", 'r') as f:
    SYSTEM_PROMPT = f.read()

display(Markdown(SYSTEM_PROMPT))

You are a search agent tasked with answering questions about the AI Engineer Europe 2026 Conference.

You have access to different context retrieval tools to help you answer user queries.

Before answering a question decide whether or not you need to retrieve additional context to answer the question correctly.
If the retrieved context does not contain relevant information to answer the query, say that you don't know. 

## Local filesystem (`session_data`)

The conference sessions are available under under `../data/session_data/`. One file per session.

File structure:
```
../
data/
 └── session_data/
  ├── workshop/
  │ ├── <title>.txt
  │ └── ... 
  ├── ...
  └── expo_session/
```

File Sample:
```
# <Title>

- **Day:** <Date of the session>
- **Time:** <Time slot of the session>
- **Room:** <Room where the session takes place>
- **Type:** <One of 'keynote', 'workshop', 'talk', 'track_keynote', 'lightning', 'expo_session'>
- **Speakers:** <Name(s) of the speaker(s)>

<Description>
```

### Define built-in shell tool

Let's define a built-in [Shell tool](https://docs.langchain.com/oss/python/integrations/tools/bash) the agent can use to execute commands in a terminal. A common use case for this is letting the LLM interact with your local file system.

<div class="alert alert-block alert-warning">
Giving agents access to the shell is powerful but also risky when used outside of a sandboxed environment because it doesn't  have any safeguards by default. <b>Use at your own risk.</b>

Note: Shell tool does not work with Windows OS.
</div>

In [24]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

# Test the shell tool
print(shell_tool.run({"commands": ["echo 'Hello World!'"]}))

Executing command:
 ["echo 'Hello World!'"]
Hello World!



/Users/leonie/Documents/code/workshop-agentic-search/.venv/lib/python3.14/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


### Create the agent

Now assemble your agent with all the components including the built-in shell tool.

In [25]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT,
    tools=[shell_tool],
)

## Run the agent

In [26]:
input_message = {
    "role": "user",
    "content": ("Are there any sessions about GEPA?"),
}

for step in agent.stream(
    {"messages": [input_message]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Are there any sessions about GEPA?
================================== Ai Message ==================================
Tool Calls:
  terminal (call_w0Yi8vNRxFy8EgbJsvBOqSJI)
 Call ID: call_w0Yi8vNRxFy8EgbJsvBOqSJI
  Args:
    commands: ['ls -R ../data/session_data | head', 'grep -Ril "GEPA" ../data/session_data | head -n 50']
Executing command:
 ['ls -R ../data/session_data | head', 'grep -Ril "GEPA" ../data/session_data | head -n 50']
================================= Tool Message =================================
Name: terminal

expo_session
keynote
lightning
talk
track_keynote
workshop

../data/session_data/expo_session:
Comprehend First, Code Later The AI Skill I Rely On Daily.txt
Expo Session – AMA AIE Europe with swyx.txt
../data/session_data/workshop/Playground in Prod - Optimising Agents in Production Environments.txt



/Users/leonie/Documents/code/workshop-agentic-search/.venv/lib/python3.14/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================== Ai Message ==================================
Tool Calls:
  terminal (call_s1QbL16Xg8p1RRxNhciTys9F)
 Call ID: call_s1QbL16Xg8p1RRxNhciTys9F
  Args:
    commands: ['grep -Ril "GEPA" ../data/session_data']
Executing command:
 ['grep -Ril "GEPA" ../data/session_data']
================================= Tool Message =================================
Name: terminal

../data/session_data/workshop/Playground in Prod - Optimising Agents in Production Environments.txt



/Users/leonie/Documents/code/workshop-agentic-search/.venv/lib/python3.14/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================== Ai Message ==================================
Tool Calls:
  terminal (call_WpzxO7kadUuXmZLpfUvRt66E)
 Call ID: call_WpzxO7kadUuXmZLpfUvRt66E
  Args:
    commands: ['cat "../data/session_data/workshop/Playground in Prod - Optimising Agents in Production Environments.txt" | sed -n \'1,200p\'']
Executing command:
 ['cat "../data/session_data/workshop/Playground in Prod - Optimising Agents in Production Environments.txt" | sed -n \'1,200p\'']
================================= Tool Message =================================
Name: terminal

# Playground in Prod - Optimising Agents in Production Environments

- **Day:** April 8
- **Time:** 10:40am-12:00pm
- **Room:** Westminster
- **Type:** workshop
- **Speakers:** Samuel Colvin
**Description:**
Deploying an agent is just the beginning. The real challenge is making it better once it's live — without redeploying, without downtime, and ideally without a human in the loop at all.

In this talk I'll introduce two

/Users/leonie/Documents/code/workshop-agentic-search/.venv/lib/python3.14/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================== Ai Message ==================================

Yes—there is at least one session that mentions **GEPA**:

- **Playground in Prod - Optimising Agents in Production Environments** (workshop)  
  - **Day/Time:** April 8, 10:40am–12:00pm  
  - **Room:** Westminster  
  - **Speaker:** Samuel Colvin  
  - **Where GEPA appears:** The session includes **“Autonomous optimisation with GEPA”** and explains how GEPA can be wired into Logfire’s managed variables to create an optimisation loop that improves agent prompts based on production traces.


In [27]:
input_message = {
    "role": "user",
    "content": ("Which sessions discuss handling regulatory constraints?"),
}

for step in agent.stream(
    {"messages": [input_message]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which sessions discuss handling regulatory constraints?
================================== Ai Message ==================================
Tool Calls:
  terminal (call_b57oFP6PhsfwyUEepsuXQvCV)
 Call ID: call_b57oFP6PhsfwyUEepsuXQvCV
  Args:
    commands: ['ls -R ../data/session_data | head', 'find ../data/session_data -type f | wc -l', 'grep -Ril "regulat" ../data/session_data | head -n 20', 'grep -Ril "compliance\\|constraint\\|constraints\\|GDPR\\|governance" ../data/session_data | head -n 20']
Executing command:
 ['ls -R ../data/session_data | head', 'find ../data/session_data -type f | wc -l', 'grep -Ril "regulat" ../data/session_data | head -n 20', 'grep -Ril "compliance\\|constraint\\|constraints\\|GDPR\\|governance" ../data/session_data | head -n 20']
================================= Tool Message =================================
Name: terminal

expo_session
keynote
lightning
talk
track_keynote
wor

/Users/leonie/Documents/code/workshop-agentic-search/.venv/lib/python3.14/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================== Ai Message ==================================
Tool Calls:
  terminal (call_pZOw8B9xrsHe9462KzZEhS1D)
 Call ID: call_pZOw8B9xrsHe9462KzZEhS1D
  Args:
    commands: ['grep -Ril "regulat\\|compliance\\|GDPR\\|sovereign\\|sovereignty\\|constraint" ../data/session_data | head -n 50', 'grep -Ril "regulat\\|compliance\\|GDPR\\|constraint\\|constraints" ../data/session_data/talk ../data/session_data/keynote ../data/session_data/workshop ../data/session_data/track_keynote ../data/session_data/lightning ../data/session_data/expo_session 2>/dev/null | head -n 50']
Executing command:
 ['grep -Ril "regulat\\|compliance\\|GDPR\\|sovereign\\|sovereignty\\|constraint" ../data/session_data | head -n 50', 'grep -Ril "regulat\\|compliance\\|GDPR\\|constraint\\|constraints" ../data/session_data/talk ../data/session_data/keynote ../data/session_data/workshop ../data/session_data/track_keynote ../data/session_data/lightning ../data/session_data/expo_session 2>/dev/null | h

/Users/leonie/Documents/code/workshop-agentic-search/.venv/lib/python3.14/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================== Ai Message ==================================
Tool Calls:
  terminal (call_SqWPOrRycbG0wwkv2ryWeIYZ)
 Call ID: call_SqWPOrRycbG0wwkv2ryWeIYZ
  Args:
    commands: ['sed -n \'1,200p\' "../data/session_data/talk/Engineering AI Systems Under Sovereignty Constraints.txt"', 'sed -n \'1,160p\' "../data/session_data/keynote/Gemma, DeepMind\'s Family of Open Models.txt"', 'sed -n \'1,160p\' "../data/session_data/talk/Connecting the Dots with Context Graphs.txt"']
Executing command:
 ['sed -n \'1,200p\' "../data/session_data/talk/Engineering AI Systems Under Sovereignty Constraints.txt"', 'sed -n \'1,160p\' "../data/session_data/keynote/Gemma, DeepMind\'s Family of Open Models.txt"', 'sed -n \'1,160p\' "../data/session_data/talk/Connecting the Dots with Context Graphs.txt"']
================================= Tool Message =================================
Name: terminal

# Engineering AI Systems Under Sovereignty Constraints

- **Day:** April 9
- **Time:** 12:4

/Users/leonie/Documents/code/workshop-agentic-search/.venv/lib/python3.14/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================== Ai Message ==================================

The sessions that discuss handling **regulatory constraints** (including sovereignty/jurisdiction/legal boundaries) are:

- **“Engineering AI Systems Under Sovereignty Constraints”** (Talk) — April 9, 12:40–1:00pm, Room St. James  
  *Focus:* how regulatory/jurisdictional constraints affect AI architecture, including data localization, auditability, transparency, and governance by design.

If you want, I can also list sessions that mention related topics like *compliance*, *GDPR*, or *governance* specifically (the dataset only clearly surfaced the one above under regulatory/constraint language).


## Set up agent with shell tool (and semantic `grep` alternative)

`grep` is based on regular expressions and is great for pattern matching. However, it has the limitation that it's not great at semantic search.

There are many alternatives to `grep` available that enable semantic search over files, for example [`jina-grep-cli`](https://github.com/jina-ai/jina-grep-cli).

### 1. Install the `jina-grep` CLI

Install the `jina-grep` CLI in your virtual environment:

```bash
git clone https://github.com/jina-ai/jina-grep-cli.git && cd jina-grep-cli
uv pip install -e .
```

### 2. Adjust the agent's system prompt

Next, you need to tell your agent about the new CLI it can use in its system prompt. Make sure to also add some explanations on how to use the `jina-grep` CLI.

In [28]:
with open("system_prompt_fs_jina_grep.md", 'r') as f:
    SYSTEM_PROMPT = f.read()

print(SYSTEM_PROMPT)

You are a search agent tasked with answering questions about the AI Engineer Europe 2026 Conference.

You have access to different context retrieval tools to help you answer user queries.

Before answering a question decide whether or not you need to retrieve additional context to answer the question correctly.
If the retrieved context does not contain relevant information to answer the query, say that you don't know. 

## Local filesystem (`session_data`)

The conference sessions are available under under `../data/session_data/`. One file per session.

File structure:
```
../
data/
 └── session_data/
  ├── workshop/
  │ ├── <title>.txt
  │ └── ... 
  ├── ...
  └── expo_session/
```

File Sample:
```
# <Title>

- **Day:** <Date of the session>
- **Time:** <Time slot of the session>
- **Room:** <Room where the session takes place>
- **Type:** <One of 'keynote', 'workshop', 'talk', 'track_keynote', 'lightning', 'expo_session'>
- **Speakers:** <Name(s) of the speaker(s)>

<Description>
``

In [29]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT,
    tools=[shell_tool],
)

In [30]:
input_message = {
    "role": "user",
    "content": ("Which sessions discuss handling regulatory constraints?"),
}

for step in agent.stream(
    {"messages": [input_message]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which sessions discuss handling regulatory constraints?
================================== Ai Message ==================================
Tool Calls:
  terminal (call_6OrqroK96CcV0dzW9I8RqVX6)
 Call ID: call_6OrqroK96CcV0dzW9I8RqVX6
  Args:
    commands: ['ls -R ../data/session_data | head', 'jina-grep -r --top-k 10 "regulatory constraints" ../data/session_data']
Executing command:
 ['ls -R ../data/session_data | head', 'jina-grep -r --top-k 10 "regulatory constraints" ../data/session_data']


/Users/leonie/Documents/code/workshop-agentic-search/.venv/lib/python3.14/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================= Tool Message =================================
Name: terminal

expo_session
keynote
lightning
talk
track_keynote
workshop

../data/session_data/expo_session:
Comprehend First, Code Later The AI Skill I Rely On Daily.txt
Expo Session – AMA AIE Europe with swyx.txt
Loading model... done
../data/session_data/talk/Engineering AI Systems Under Sovereignty Constraints.txt:1:# Engineering AI Systems Under Sovereignty Constraints
- **Day:** April 9
- **Time:** 12:40-1:00pm
- **Room:** St. James
- **Type:** talk
- **Track:** Context Engineering
- **Speakers:** Bilge Yücel
**Description:**
Regulatory and jurisdictional constraints are no longer an edge case in AI system design; they now influence architecture decisions as much as model quality. From European Parliament discussions around prioritising non-US technology in public procurement and building a “Eurostack,” to the launch of the AWS European Sovereign Cloud, sovereignty is becoming a concrete technical 